# Train Amharic Transliterator From Clean CSV (Colab)

This notebook trains from one clean canonical CSV (`rom2amh` or `amh2rom`) using the same Seq2Seq approach as the project trainer, but **without** pushing to Hugging Face Hub.

It saves the trained model locally and creates a zip file for download.

## 1) Install Dependencies

Run this first in Colab to install required packages (`transformers`, `datasets`, `evaluate`, etc.).

In [ ]:
# Install dependencies in Colab
!pip -q install transformers datasets evaluate sacrebleu jiwer sentencepiece accelerate pandas scikit-learn

## 2) Mount Google Drive

Run this once so the notebook can read your dataset directly from Drive.

## 3) Imports And Config

This block imports libraries and defines all run settings. Update `CLEAN_CSV_PATH` and `TASK_DIRECTION` before training.

In [4]:
import os
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# =========================
# USER CONFIG (edit these)
# =========================
CLEAN_CSV_PATH = "/kaggle/input/datasets/ebaadisukenea/amh-to-rom-clean/amharic_clean_merged.csv"  # Kaggle dataset path
TASK_DIRECTION = "rom2amh"  # Only used when a 'direction' column exists

MODEL_CHECKPOINT = "google/mt5-small"  # Better multilingual base for Amharic
# Kaggle: writable output lives under /kaggle/working (not /content)
OUTPUT_DIR = "/kaggle/working/amharic_translit_model"
MODEL_NAME = f"amharic-translit-{TASK_DIRECTION}"

TEST_SIZE = 0.1
RANDOM_STATE = 42
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 1

# Optional subset for quick smoke test
USE_SUBSET = True
SUBSET_ROWS = 1000

print("Config loaded.")

Config loaded.


## 4) Prepare Data For Training

This block loads your clean CSV, enforces the selected direction, creates `source` and `target`, removes empty rows, and performs a train/test split.

In [5]:
# Load CSV and build source/target
assert os.path.exists(CLEAN_CSV_PATH), f"File not found: {CLEAN_CSV_PATH}"

df = pd.read_csv(CLEAN_CSV_PATH)
cols = set(df.columns)

# Case A: already training-ready merged file with source/target
if {"source", "target"}.issubset(cols):
    train_ready_df = df[["source", "target"]].copy()

# Case B: canonical file with amharic_text/romanized_text (and optional direction)
elif {"amharic_text", "romanized_text"}.issubset(cols):
    if "direction" in df.columns:
        df = df[df["direction"] == TASK_DIRECTION].copy()

    if TASK_DIRECTION == "rom2amh":
        # Romanized -> Amharic
        df["source"] = df["romanized_text"].astype(str)
        df["target"] = df["amharic_text"].astype(str)
    elif TASK_DIRECTION == "amh2rom":
        # Amharic -> Romanized
        df["source"] = df["amharic_text"].astype(str)
        df["target"] = df["romanized_text"].astype(str)
    else:
        raise ValueError("TASK_DIRECTION must be 'rom2amh' or 'amh2rom'")

    train_ready_df = df[["source", "target"]].copy()
else:
    raise ValueError(
        "Input CSV must contain either ['source','target'] or ['amharic_text','romanized_text']."
    )

if USE_SUBSET:
    train_ready_df = train_ready_df.sample(
        n=min(SUBSET_ROWS, len(train_ready_df)), random_state=RANDOM_STATE
    ).copy()

# Drop empty rows
before = len(train_ready_df)
train_ready_df = train_ready_df[
    (train_ready_df["source"].astype(str).str.strip() != "")
    & (train_ready_df["target"].astype(str).str.strip() != "")
].copy()
after = len(train_ready_df)
print(f"Rows before cleanup: {before}, after cleanup: {after}")

train_df, test_df = train_test_split(
    train_ready_df[["source", "target"]], test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
raw_datasets = DatasetDict({"train": train_dataset, "test": test_dataset})

Rows before cleanup: 1000, after cleanup: 1000
Train rows: 900 | Test rows: 100


## 4) Train The Seq2Seq Model

This block follows the same training style as your project trainer: tokenize `source/target`, train with `Seq2SeqTrainer`, and evaluate with SacreBLEU. Hugging Face Hub publishing is disabled (`push_to_hub=False`).

In [6]:
# Tokenizer/model + training (same approach as train_multi_transliterator.py)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    inputs = [s for s in examples["source"]]
    targets = [t for t in examples["target"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    # Generated ids can include invalid values for Rust decode -> OverflowError.
    # Clamp to valid vocab range and use int64 before batch_decode.
    preds = np.asarray(preds)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    vocab_upper = max(len(tokenizer) - 1, 0)
    preds = np.where(preds < 0, pad_id, preds)
    preds = np.clip(preds, 0, vocab_upper).astype(np.int64)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": round(result["score"], 4)}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = round(float(np.mean(prediction_lens)), 4)
    return result

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    save_total_limit=3,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    push_to_hub=False,  # IMPORTANT: do not publish
    report_to="none",
    dataloader_pin_memory=False,  # avoids pin_memory warning on CPU / some setups
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

...

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,No log,21.309580,0.067400,3.550000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 21.309579849243164, 'eval_bleu': 0.0674, 'eval_gen_len': 3.55, 'eval_runtime': 53.2405, 'eval_samples_per_second': 1.878, 'eval_steps_per_second': 0.131, 'epoch': 1.0}


## 6) Export Model Artifacts

This final block saves the trained model/tokenizer locally, compresses them into a zip file, and triggers Colab download.

In [7]:
# Save model/tokenizer + create downloadable zip
final_export_dir = f"{OUTPUT_DIR}_final"
os.makedirs(final_export_dir, exist_ok=True)

trainer.save_model(final_export_dir)
tokenizer.save_pretrained(final_export_dir)

zip_base = os.path.join("/kaggle/working", f"{MODEL_NAME}_export")
zip_path = shutil.make_archive(zip_base, "zip", final_export_dir)
print(f"Model export folder: {final_export_dir}")
print(f"Zip created: {zip_path}")

# Colab: browser download. Kaggle: use Output tab / copy from /kaggle/working
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Saved zip (Kaggle): {zip_path}")
except Exception as e:
    print("Auto-download skipped.")
    print(f"Manual path: {zip_path}")
    print(f"Reason: {e}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model export folder: /kaggle/working/amharic_translit_model_final
Zip created: /kaggle/working/amharic-translit-rom2amh_export.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7) Test The Trained Model In Colab

Use this section after training/export to run quick transliteration checks and compare predictions against sample test rows.

In [8]:
# Simple inference test block
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_DIR = f"{OUTPUT_DIR}_final"

tokenizer_infer = AutoTokenizer.from_pretrained(MODEL_DIR)
model_infer = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_infer.to(device)
model_infer.eval()

def transliterate(text, max_new_tokens=128):
    inputs = tokenizer_infer(text, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        out = model_infer.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer_infer.decode(out[0], skip_special_tokens=True)

# rom -> amh (prefix required)
print("ROM2AMH:", transliterate("<to_atrs>selam new?"))

# amh -> rom (prefix required)
print("AMH2ROM:", transliterate("<to_abtrs>ሰላም ነው?"))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


ROM2AMH: <extra_id_0>
AMH2ROM: <extra_id_0>


In [9]:
# Quick sample evaluation on held-out test rows
sample_df = test_df.sample(min(10, len(test_df)), random_state=42)

for _, row in sample_df.iterrows():
    pred = transliterate(row["source"])
    print("SRC :", row["source"])
    print("TGT :", row["target"])
    print("PRED:", pred)
    print("-" * 80)

SRC : <to_atrs>sile yehaseti rasishi ra’iyochi yayeshina mwariti yamwaretishi yewisheti bīhonimi k’enachewi yemech’eresha , k’it’ati yemīk’ebelubeti bederesebachewi gīzē maletimi kifuwochi sewochi layi bemīgedeluti tikemerīyaleshi .
TGT : ስለ የሃሰት ራስሽ ራእዮች ያየሽና ሟርት ያሟረትሽ የውሸት ቢሆንም ቀናቸው የመጨረሻ , ቅጣት የሚቀበሉበት በደረሰባቸው ጊዜ ማለትም ክፉዎች ሰዎች ላይ በሚገደሉት ትከመሪያለሽ .
PRED: <extra_id_0> .
--------------------------------------------------------------------------------
SRC : <to_atrs>ke’inesu ānidu inidetefewese baye gīzē āmilakini betalak’i dimits’i iyamesegene temelese .
TGT : ከእነሱ አንዱ እንደተፈወሰ ባየ ጊዜ አምላክን በታላቅ ድምጽ እያመሰገነ ተመለሰ .
PRED: <extra_id_0>.
--------------------------------------------------------------------------------
SRC : <to_atrs>dek’e mezamuritu fits’umi bayihonumi inikwa be’inesu layi yitemameni sileneberi ke’isu yemībelit’i sira inidemīyakenawinu negirwachewali .
TGT : ደቀ መዛሙርቱ ፍጹም ባይሆኑም እንኳ በእነሱ ላይ ይተማመን ስለነበር ከእሱ የሚበልጥ ስራ እንደሚያከናውኑ ነግሯቸዋል .
PRED: <extra_id_0> .
--------------------------